# 01 — Molecular geometry review

Review molecular masks and their registration to anatomy. Each registration route is shown separately because it comes from a different experiment step.

In [ ]:
# 02 — Choose the fish and saved run
from pathlib import Path
from codeants_2pf_hcr.qc_notebooks import QCNotebookConfig, resolve_qc_notebook_context

FISH_ID = "L765_f04"
LOCAL_ROOT = Path("/Volumes/dataDrive/dataProcessing/2p_processing")
PIPELINE_ROOT = Path("/Volumes/dataDrive/dataProcessing/2p_processing/_staged_pipeline_runs/20260803T112035Z_functional_registration_ants_no_fallback/L765_f04")  # Molecular masks are reviewed from the fish-local aligned directory below.
CONTEXT = resolve_qc_notebook_context(QCNotebookConfig(FISH_ID, LOCAL_ROOT, PIPELINE_ROOT))


## 03 — Choose the files to review

Only change these paths when the chosen run uses different names. Do not combine files from different runs. An empty list means no file has been selected for that track.

In [ ]:
# 04 — Locate registration and mask files
TRACK_ARTIFACTS = {
    "rbest_to_in_vivo": [CONTEXT.fish_dir / "02_reg" / "01_rbest-2p"],
    "rbest_to_ex_vivo": [CONTEXT.fish_dir / "02_reg" / "03_rbest-exvivo"],
    "ex_vivo_to_in_vivo": [CONTEXT.fish_dir / "02_reg" / "04_exvivo-2p"],
    "later_round_to_ex_vivo": [CONTEXT.fish_dir / "02_reg" / "05_rn-exvivo"],
}
from codeants_2pf_hcr.plots.qc_molecular import resolve_hcr_matching_artifact_root
HCR_ALIGNED_ROOT = resolve_hcr_matching_artifact_root(staged_root=CONTEXT.stage_paths["register-hcr-to-anatomy"] / "confocal" / "aligned", fish_dir=CONTEXT.fish_dir)  # One persisted artifact root only; fish-local fallback supports legacy controls.
EXCLUDED_LABEL_TOKENS = ("gad2", "slc17a6b", "_r4_")  # Fish-specific scientific exclusions; adjust explicitly for another fish.
RETAINED_LABEL_PATHS = [
    path for path in sorted(HCR_ALIGNED_ROOT.glob("*_cp_masks_in_2p_labels_uint16.tif"))
    if not path.name.startswith("._") and not any(token in path.name.lower() for token in EXCLUDED_LABEL_TOKENS)
]
PAIR_CANDIDATES = None  # Candidate matching has not been run yet.


## 05 — Cross-modality mask-size review

Bounding boxes are measured from persisted labels on the common anatomy grid. The Q05 XY threshold removes only implausibly small masks from the distributions; Q95 XY masks remain displayed and flagged for review.

In [ ]:
# 06 — Inspect physical bounding-box sizes without altering masks
from IPython.display import display
from codeants_2pf_hcr.plots.qc_molecular import build_cross_modality_mask_size_table, plot_cross_modality_mask_sizes

ANATOMY_LABELS_PATH = CONTEXT.fish_dir / "03_analysis" / "structural" / "cp_masks" / f"{FISH_ID}_anatomy_00001_uint8_8bit_cp_masks.tif"
REFERENCE_ANATOMY_PATH = CONTEXT.fish_dir / "02_reg" / "00_preprocessing" / "2p_anatomy" / f"{FISH_ID}_anatomy_2P_GCaMP.nrrd"
FUNCTIONAL_LABEL_PATHS = [path for path in sorted((CONTEXT.stage_paths["transform-functional-rois-to-anatomy"] / "functional" / "anatomy").glob("*_func_mask_in_2p.tif")) if not path.name.startswith("._")]
MASK_SIZE_QC = build_cross_modality_mask_size_table(
    anatomy_labels_path=ANATOMY_LABELS_PATH,
    functional_label_paths=FUNCTIONAL_LABEL_PATHS,
    hcr_label_paths=RETAINED_LABEL_PATHS,
    reference_anatomy_path=REFERENCE_ANATOMY_PATH,
)
MASK_SIZE_QC_SUMMARY = MASK_SIZE_QC.groupby("modality").agg(n_before=("label", "size"), n_after_q05_xy=("included_in_distribution", "sum"), q05_xy_um=("xy_q05_um", "first"), q95_xy_um=("xy_q95_um", "first"), q05_drops=("is_xy_hard_drop", "sum"), q95_flags=("is_xy_q95_flag", "sum"))
display(MASK_SIZE_QC_SUMMARY)
plot_cross_modality_mask_sizes(MASK_SIZE_QC, fish_id=FISH_ID);

In [ ]:
# 07 — Summarize molecular geometry
from IPython.display import display
from codeants_2pf_hcr.plots.qc_molecular import inspect_molecular_geometry_qc, plot_molecular_geometry_qc

report = inspect_molecular_geometry_qc(
    fish_id=FISH_ID,
    pipeline_root=CONTEXT.pipeline_root,
    fish_dir=CONTEXT.fish_dir,
    track_artifacts=TRACK_ARTIFACTS,
    segmentation_table_path=None,
    segmentation_label_paths=RETAINED_LABEL_PATHS,
    pair_candidates_path=PAIR_CANDIDATES,
    pair_candidates_state="not_started",
)
display(report["issues"], report["tracks"], report["label_masks"], report["pair_ambiguity"])
plot_molecular_geometry_qc(report);

## 08 — Interactive anatomy-space HCR label review

The viewer displays only the explicitly retained labels in the in-vivo anatomy grid. It is a read-only placement review and does not create candidate matches or assign identity.

In [ ]:
# 09 — Review retained HCR labels over in-vivo anatomy
import matplotlib.pyplot as plt
plt.switch_backend("module://ipympl.backend_nbagg")
from codeants_2pf_hcr.plots.qc_molecular import load_molecular_label_viewer, load_hcr_match_overlay_viewer, show_molecular_label_viewer

ANATOMY_PATH = CONTEXT.fish_dir / "02_reg" / "00_preprocessing" / "2p_anatomy" / f"{FISH_ID}_anatomy_2P_GCaMP.nrrd"
viewer_data = load_molecular_label_viewer(ANATOMY_PATH, RETAINED_LABEL_PATHS)
show_molecular_label_viewer(viewer_data)

# Optional persisted accepted/rejected toggle; this classifies only saved label IDs and never updates pairs.
HCR_PAIR_ROOT = HCR_ALIGNED_ROOT
HCR_REVIEW_PATHS = [HCR_PAIR_ROOT / path.name.replace("_labels_uint16.tif", "_review.csv") for path in RETAINED_LABEL_PATHS]
HCR_FINAL_PAIR_PATHS = [HCR_PAIR_ROOT / path.name.replace("_labels_uint16.tif", "_final_pairs.csv") for path in RETAINED_LABEL_PATHS]
match_overlay_data = load_hcr_match_overlay_viewer(ANATOMY_PATH, RETAINED_LABEL_PATHS, HCR_REVIEW_PATHS, HCR_FINAL_PAIR_PATHS)
show_molecular_label_viewer(match_overlay_data)

## 10 — Decide what needs review

Review mask coverage, each registration route, and uncertain molecular matches. A file being present does not mean it has been accepted.

## 11 — Persisted HCR-to-anatomy matching flow

Each gene/round uses the legacy two-ring donut grammar. The inner ring is final one-to-one acceptance; the outer ring partitions all segmented labels by their persisted terminal matching outcome. Counts are read from saved labels, review rows, and final-pair artifacts only.

In [ ]:
# 12 — Audit HCR functional-plane representation per retained gene/round
from IPython.display import display
from codeants_2pf_hcr.plots.qc_molecular import build_hcr_functional_plane_status_donut_table, plot_hcr_functional_plane_status_donuts

HCR_PAIR_ROOT = HCR_ALIGNED_ROOT
HCR_ACTIVITY_STATUS_PATH = CONTEXT.fish_dir / "03_analysis" / "functional" / "registration" / "hcr_activity_status.csv"
HCR_FUNCTIONAL_PLANE_DONUTS = build_hcr_functional_plane_status_donut_table(label_paths=RETAINED_LABEL_PATHS, activity_status_path=HCR_ACTIVITY_STATUS_PATH)
display(HCR_FUNCTIONAL_PLANE_DONUTS)
plot_hcr_functional_plane_status_donuts(HCR_FUNCTIONAL_PLANE_DONUTS, fish_id=FISH_ID);